In [ ]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import jv, j0
import pandas as pd

In [ ]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [ ]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [ ]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [ ]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [ ]:
if __name__ == "__main__":
    main()

s_lst = []
for key, value in enumerate(sqrt_s_lst):
    s_lst.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {s_lst[key]:.2f} GeV^2")

In [ ]:
def eikonal_int(b, q, amp_born):
    return  q * jv(0, q * b) * amp_born

def amp_eik_int(b, q, eik):
    return b * jv(0, q * b) * (1 - np.exp(eik * 1j))

eikonal_int_lst = []
eikonal_lst = []

amp_eik_int_lst = []
amp_eik_lst = []

b_lst = list(np.linspace(0, 30, 130))  
q_lst = list(np.linspace(0, 30, 130))  

# amp_eik_int_value, _ = fixed_quad(
#     lambda b: amp_eik_int(b, 0, eikonal_value), 0, 30, n = 1000
# )
# amp_eik_int_lst.append(amp_eik_int_value)
# amp_eik_value = amp_eik_int_value * 1j * s_value
# amp_eik_lst.append(amp_eik_value.imag)
# # print(amp_eik_value.imag)




In [ ]:
# lst_amp_eik = []
# lst_sigma_tot = []
# n = 10

# def sigma_tot_eik(amp, s):
#     return (4 * np.pi / s) * amp.imag * 0.389379323

# def all_integration(b, q , amp_born, s):
#     return 1j * s * b * jv(0, q*b) * (1 - np.exp(1j* (1/s * q * jv(0, b* q) * amp_born)))


# for idx, (amp_born_value, s_value) in enumerate(zip(amp_born_lst, s_lst)):

#     if idx < len(s_lst) - 1:  

#         def inner_integral(q):
#             return fixed_quad(
#                 lambda b: all_integration(b, q, amp_born_value, s_value), 
#                 0, 10, 
#                 n = n
#             )[0]
        
#         amp_eik_value = fixed_quad(
#             inner_integral,
#             0, 10,
#             n = n
#         )[0]


#         lst_amp_eik.append(amp_eik_value)

#         print(f'amp born = {amp_born_value} - s = {s_value}')
#         print(f'amp eikonalizada = {amp_eik_value}')
#         print('\n')
        
        

#         # sigma_tot_eik_value = sigma_tot_eik(amp_eik_value, s_value)
#         # lst_sigma_tot.append(sigma_tot_eik_value)
#         # print(f'sigma tot eikonalizada = {sigma_tot_eik_value}')
#         # print('\n')

#     else: pass



In [25]:
from scipy.integrate import quad
from scipy.special import jv
import numpy as np

def sigma_tot_eik(amp, s):
    """Calculate total cross section in mb"""
    return (4 * np.pi / s) * amp.imag * 0.389379323  # GeV^-2 to mb conversion

def chi(s, b, amp_born):
    """Calculate eikonal function χ(s,b)"""
    # Integrate over q (with t=-q²)
    integrand = lambda q: q * jv(0, b*q) * amp_born
    result, _ = quad(integrand, 0, 30)
    return result / s

def A_eik(s, amp_born, q_max=10, b_max=10):
    """Calculate eikonalized amplitude at t=0"""
    # For forward amplitude (t=0 → q=0)
    integrand = lambda b: b * (1 - np.exp(1j * chi(s, b, amp_born)))
    result, _ = quad(integrand, 0, 30)
    return 1j * s * result

# Example usage:
lst_amp_eik = []
lst_sigma_tot = []

for s_value, amp_born_value in zip(s_lst, amp_born_lst):
    # Define your Born amplitude as a function of t
    def amp_born_func(t):
        # Example implementation - replace with your actual Born amplitude
        return 10.0 * np.exp(5.0 * t)  # Should return complex value
    
    # Calculate eikonalized amplitude
    amp_eik_value = A_eik(s_value, amp_born_value)
    lst_amp_eik.append(amp_eik_value)
    
    # Calculate total cross section
    sigma_tot = sigma_tot_eik(amp_eik_value, s_value)
    lst_sigma_tot.append(sigma_tot)
    
    print(f"s = {s_value} GeV²")
    print(f"Eikonalized amplitude: {amp_eik_value}")
    print(f"Total cross section: {sigma_tot} mb\n")

s = 1 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 10201 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 40401 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 90601 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 160801 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 251001 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 361201 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 491401 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 641601 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 811801 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 1002001 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 1212201 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 1442401 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 1692601 GeV²
Eikonalized amplitude: 0j
Total cross section: 0.0 mb

s = 196280

/home/victorli/miniconda3/lib/python3.12/site-packages/scipy/integrate/_quadpack_py.py:606: ComplexWarning:

Casting complex values to real discards the imaginary part



In [ ]:
#gpt/deep

from scipy.integrate import quad
from scipy.special import jv
import numpy as np

def sigma_tot_eik(amp, s):
    """Calculate total cross section in mb"""
    return (4 * np.pi / s) * amp.imag * 0.389379323  # GeV^-2 to mb conversion

def chi(s, b, amp_born_func):
    """Calculate eikonal function χ(s,b)"""
    # Integrate over q (with t=-q²)
    integrand = lambda q: q * jv(0, b*q) * amp_born_func(-q**2)
    result, _ = quad(integrand, 0, 30)
    return result / s

def A_eik(s, amp_born_func, q_max=10, b_max=10):
    """Calculate eikonalized amplitude at t=0"""
    # For forward amplitude (t=0 → q=0)
    integrand = lambda b: b * (1 - np.exp(1j * chi(s, b, amp_born_func)))
    result, _ = quad(integrand, 0, 30)
    return 1j * s * result

# Example usage:
lst_amp_eik = []
lst_sigma_tot = []

for s_value in s_lst:
    # Define your Born amplitude as a function of t
    def amp_born_func(t):
        # Example implementation - replace with your actual Born amplitude
        return 10.0 * np.exp(5.0 * t)  # Should return complex value
    
    # Calculate eikonalized amplitude
    amp_eik_value = A_eik(s_value, amp_born_func)
    lst_amp_eik.append(amp_eik_value)
    
    # Calculate total cross section
    sigma_tot = sigma_tot_eik(amp_eik_value, s_value)
    lst_sigma_tot.append(sigma_tot)
    
    print(f"s = {s_value} GeV²")
    print(f"Eikonalized amplitude: {amp_eik_value}")
    print(f"Total cross section: {sigma_tot} mb\n")

In [ ]:
# def eikonal_int(b, q, amp_born):
#     return  q * jv(0, q * b) * amp_born

# def amp_eik_int(b, q, eik):
#     return b * jv(0, q * b) * (1 - np.exp(eik * 1j))

# eikonal_int_lst = []
# eikonal_lst = []

# amp_eik_int_lst = []
# amp_eik_lst = []

# b_lst = list(np.linspace(0, 30, 130))  
# q_lst = list(np.linspace(0, 30, 130))  

# for idx, (q_value, b_value, amp_born_value, s_value) in enumerate(zip(q_lst, b_lst, amp_born_lst, s_lst)):
#     if idx < len(q_lst) - 1:  

#         next_q = q_lst[idx+1]
#         next_b = b_lst[idx+1]

#         eikonal_int_value, _= fixed_quad(
#             lambda q: eikonal_int(b_value, q, amp_born_value), q_value, next_q, n = 1000
#         )
#         print(f'lower = {q_value}, upper = {next_q}')
#         print(f'amp = {amp_born_value}, s = {s_value}')
#         print(f'eikonal integral value = {eikonal_int_value}')
#         eikonal_value = eikonal_int_value/s_value
#         print(f'eikonal value = {eikonal_value}')
#         print('\n')

#         amp_eik_int_value, _ = fixed_quad(
#             lambda b: amp_eik_int(b, next_q, eikonal_value), 
#             b_value, next_b, n = 1000
#         )
#         print(f'amp eik integral value = {amp_eik_int_value}')
#         amp_eik_value = 1j * s_value * amp_eik_int_value
#         print(f'amp eik = {amp_eik_value.imag}')
#         print('\n')
#         print(100*'-')
#         print('\n')
        

#     else:
#         pass  
    

In [ ]:
# def eikonal(b, q, amp_born, s):
#     return  (q * jv(0, q * b) * amp_born) / s

# def amp_eik(b, q, eik, s):
#     return b * jv(0, q * b) * (1 - np.exp(eik * 1j)) * 1j* s

# result_lst = []
# for idx, (b_value, amp_born_value, s_value) in enumerate(zip(b_lst, amp_born_lst, s_lst)):
        
#     result, _ = fixed_quad(
#         lambda b: amp_eik(
#             b,
#             q,  
#             fixed_quad(  
#                 lambda q: eikonal(b, q, amp_born_value, s_value),
#                 0, 30, n=10000
#             )[0],
#             s_value
#         ),
#         0, 30, n=10000
#     )

#     result_lst.append(result)

    
# print(result_lst)